# Causal FX forecasting: complete research walkthrough

Use this repositoryâ€™s isolated environment (`python -m pip install -e ".[dev]"`) and install Jupyter/matplotlib. Cells are initially unexecuted. Calls use production modules and preserve all historical files. Bank of Canada reference rates are not executable market quotes. Observed 2026 comparisons are retrospective; no profitability claim follows.

In [ ]:
from pathlib import Path
import os, json, subprocess, sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from its own repository'
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
RUN_FETCH = False
RUN_TRAIN = False
RUN_INFERENCE = False
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])


## Download a fixed historical development snapshot
Explicit dates prevent accidental inclusion of later observations. Production acquisition saves original bytes, hash, source terms and the current-snapshot vintage limitation. This notebook never fetches the observed 2026 test automatically.

In [ ]:
if RUN_FETCH:
    from src.data.multi_asset import fetch_snapshot
    snapshot = Path('data/external/notebook-development')
    assert not snapshot.exists(), 'Choose a fresh snapshot directory; never replace historical bytes'
    panel, provenance = fetch_snapshot(snapshot, '2017-01-01', '2024-12-31')
    print(provenance)
else:
    print('Acquisition disabled; existing production snapshots remain unchanged.')


## Inspect the causal protocol
Targets span the delayed decisionâ€™s t+1 to t+2 observations; labels at training boundaries are purged. The production scaler is fit on training only, posterior inference is filtered causally, and calibration is separated from model selection. Shared/blend/recency procedures retain zero and the v2 baseline as eligible alternatives.

In [ ]:
plan = json.loads(Path('docs/shared-forecast-plan.json').read_text(encoding='utf-8'))
print(json.dumps(plan, indent=2))


## Optional development-only training
The production runner uses its frozen historical cached paths; see the plan for the exact windows. Acquisition above illustrates a reproducible separate snapshot and does not silently replace study data. New output paths and copied plans protect prior reports. No observed-test release is called here.

In [ ]:
from datetime import datetime, timezone
if RUN_TRAIN:
    output = Path('models') / ('notebook-shared-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    assert not output.exists()
    subprocess.run([sys.executable, 'train.py', 'shared', '--output', str(output)], check=True)


## Read aggregate evidence and plot
The recorded shared/blend/recency studies rejected changes that missed the prespecified magnitude/year-consistency gate. The zero forecast remained competitive. Reading an aggregate file is not a new test evaluation.

In [ ]:
reports = list(Path('models').glob('shared*/results.json'))
print('Available aggregate reports:', [str(path) for path in reports])
for path in reports:
    value = json.loads(path.read_text(encoding='utf-8'))
    print(path, json.dumps(value, indent=2)[:6000])


In [ ]:
import matplotlib.pyplot as plt
prior = json.loads(Path('models/multi-asset-v1/evaluation.json').read_text(encoding='utf-8'))
v2 = json.loads(Path('models/uncertainty-v2/results.json').read_text(encoding='utf-8'))
values = [prior['forecast_scores']['pooled']['global_rmse'], v2['pooled_rmse'], v2['pooled_zero_rmse']]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(['Prior global', 'Retained regularized v2', 'Zero'], values)
ax.set(ylabel='RMSE (return units)', title='Previously observed 2026 aggregate comparison')
plt.tight_layout()
print('Exact values read from tracked aggregate JSON; protocol: docs/uncertainty-study.md')


## Saved-checkpoint inference on a historical input panel
Use a user-supplied complete, ordered historical panel and local ignored checkpoint. The production forecaster applies its saved causal context/scaling. This creates forecasts only and computes no accuracy, trading result or test metric.

In [ ]:
if RUN_INFERENCE:
    import joblib
    import pandas as pd
    from src.models.uncertainty import forecast
    model = Path(os.environ.get('QUANT_MODEL', 'models/uncertainty-v2/bundle.joblib'))
    panel_path = Path(os.environ.get('QUANT_PANEL', 'data/external/notebook-development/panel.csv'))
    assert model.exists() and panel_path.exists(), 'Provide local weights and a complete ordered panel'
    forecasts = forecast(joblib.load(model), pd.read_csv(panel_path))
    print('Forecast rows:', len(forecasts))
    display(forecasts.head())


## Limits
Finite-sample interval coverage is empirical, not guaranteed under changing regimes. Online labels mature with delays and saved point/scale forecasts cannot use future observations. Current API snapshots lack historical as-published vintages. The study does not change risk/trading policy, and zero forecasts remain a serious baseline.